In [49]:
import pandas as pd
import numpy as np
import ast

In [50]:
df = pd.read_csv("../Data/cleaned_meta_app.csv")

In [51]:
df.head()

,Category,Product_Name,Avg_Rating,Price,Details,Parent_ASIN
0,Industrial & Scientific,"ROVSUN Ice Maker Machine Countertop, Make 44lb...",3.7,NaN,"{'Brand': 'ROVSUN', 'Model Name': 'ICM-2005', ...",B08Z743RRD
1,Tools & Home Improvement,"HANSGO Egg Holder for Refrigerator, Deviled Eg...",4.2,NaN,"{'Manufacturer': 'HANSGO', 'Part Number': 'HAN...",B097BQDGHJ
2,Tools & Home Improvement,"Clothes Dryer Drum Slide, General Electric, Ho...",3.5,NaN,"{'Manufacturer': 'RPI', 'Part Number': 'WE1M33...",B00IN9AGAE
3,Tools & Home Improvement,154567702 Dishwasher Lower Wash Arm Assembly f...,4.5,NaN,"{'Manufacturer': 'folosem', 'Part Number': '15...",B0C7K98JZS
4,Tools & Home Improvement,Whirlpool W10918546 Igniter,3.8,25.07,"{'Manufacturer': 'Whirlpool', 'Part Number': '...",B07QZHQTVJ


In [52]:
# Cleaning and combining the features...
def parse_details(x):
    try:
        return " ".join(ast.literal_eval(x).values())
    except:
        return  ""

df["Details_Clean"] = df['Details'].apply(parse_details)

In [53]:
df["Combined"] = (df["Category"].fillna("") + " " + df["Product_Name"].fillna("") + " " + df["Details_Clean"].fillna(""))

In [54]:
print(df["Combined"].head())

0    Industrial & Scientific ROVSUN Ice Maker Machi...
1    Tools & Home Improvement HANSGO Egg Holder for...
2    Tools & Home Improvement Clothes Dryer Drum Sl...
3    Tools & Home Improvement 154567702 Dishwasher ...
4    Tools & Home Improvement Whirlpool W10918546 I...
Name: Combined, dtype: object


In [55]:
# Convert to vectors (TF-IDF)

from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1,2))
tfidf_matrix = tfidf.fit_transform(df["Combined"])

In [56]:
# Compute the similarity

from sklearn.metrics.pairwise import cosine_similarity

def similarity_based_recommend(product_name, top_n=5):

    matches = df[df["Product_Name"] == product_name]

    if len(matches) == 0:
        print("Product not found!!!")
        return
    
    idx = matches.index[0]

    sim_scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()

    top_idx = np.argpartition(sim_scores, -top_n-1)[-top_n-1:]

    top_idx = top_idx[np.argsort(-sim_scores[top_idx])]

    top_idx = top_idx[top_idx != idx][:top_n]

    results = df.iloc[top_idx][["Product_Name", "Category", "Parent_ASIN"]]

    return results

In [57]:
result = similarity_based_recommend("ROVSUN Ice Maker Machine Countertop, Make 44lbs Ice in 24 Hours, Compact & Portable Ice Maker with Ice Basket for Home, Office, Kitchen, Bar (Silver)")

In [58]:
result

,Product_Name,Category,Parent_ASIN
25597,"ROVSUN Ice Maker Machine Countertop, Make 44lb...",Industrial & Scientific,B08Z7WB1DR
26468,"ROVSUN Ice Maker Machine Countertop, Make 26lb...",Industrial & Scientific,B08Z7DRPD9
32140,"ROVSUN Ice Maker Machine Countertop, Make 26lb...",Industrial & Scientific,B08Z7T97FD
68769,"ROVSUN Ice Maker Machine Countertop, Make 26lb...",Industrial & Scientific,B08Z82KCDB
15904,"ROVSUN Countertop Ice Maker Machine, Make 40lb...",Industrial & Scientific,B08F9QS3TR


In [59]:
result["Product_Name"].iloc[1]

'ROVSUN Ice Maker Machine Countertop, Make 26lbs Ice in 24 Hours, Compact & Portable Ice Maker with Ice Basket for Home, Office, Kitchen, Bar (Silver)'

In [60]:
result["Product_Name"].iloc[2]

'ROVSUN Ice Maker Machine Countertop, Make 26lbs Ice in 24 Hours, Compact & Portable Ice Maker with Ice Basket for Home, Office, Kitchen, Bar (Black)'

In [61]:
result["Product_Name"].iloc[3]

'ROVSUN Ice Maker Machine Countertop, Make 26lbs Ice in 24 Hours, Compact & Portable Ice Maker with Ice Basket for Home, Office, Kitchen, Bar (Blue)'

In [62]:
df["Details"][0]

"{'Brand': 'ROVSUN', 'Model Name': 'ICM-2005', 'Capacity': '44 Pounds', 'Wattage': '120 watts', 'Voltage': '120 Volts', 'Package Dimensions': '18 x 16 x 13.5 inches; 26.5 Pounds', 'Date First Available': 'March 17, 2021', 'Manufacturer': 'ROVSUN'}"

In [63]:
for i in range(0, 10):
    print(df["Details"][i])

{'Brand': 'ROVSUN', 'Model Name': 'ICM-2005', 'Capacity': '44 Pounds', 'Wattage': '120 watts', 'Voltage': '120 Volts', 'Package Dimensions': '18 x 16 x 13.5 inches; 26.5 Pounds', 'Date First Available': 'March 17, 2021', 'Manufacturer': 'ROVSUN'}
{'Manufacturer': 'HANSGO', 'Part Number': 'HANSGO', 'Item Weight': '5 ounces', 'Product Dimensions': '10.6"L x 4.9"W x 2.7"H', 'Item model number': 'HH0151', 'Size': 'medium', 'Color': 'Bright white', 'Material': 'Plastic', 'Shape': 'Rectangular', 'Handle Material': 'Plastic', 'Batteries Included?': 'No', 'Batteries Required?': 'No', 'Best Sellers Rank': {'Tools & Home Improvement': 246177, 'Refrigerator Egg Trays': 288}, 'Date First Available': 'June 16, 2021', 'Brand': 'HANSGO', 'Number of Sets': '1'}
{'Manufacturer': 'RPI', 'Part Number': 'WE1M333,', 'Item Weight': '0.352 ounces', 'Package Dimensions': '5.5 x 4.7 x 0.4 inches', 'Item model number': 'WE1M333,', 'Is Discontinued By Manufacturer': 'No', 'Item Package Quantity': '1', 'Batteries

In [64]:
# Now to improve our recommendation and reduce the redundancy we choose MMR method (Maximal Marginal Relevance)...
# MMR = Relevance - Redundancy

"""
Maximal Marginal Relevance (MMR)
--------------------------------

MMR is a ranking technique used to select items that are both:
1. Relevant to a query
2. Diverse from each other

It is commonly used in recommendation systems, search engines, and information retrieval
to avoid returning near-duplicate results.

Formula:
    MMR(d) = λ * Sim(q, d) - (1 - λ) * max_{d' ∈ S} Sim(d, d')

Where:
    q  : Query item (e.g., product user interacted with)
    d  : Candidate item being evaluated
    S  : Set of already selected/recommended items

    Sim(q, d)       : Similarity between query and candidate (relevance)
    Sim(d, d')      : Similarity between candidate and selected items (redundancy)
    max_{d' ∈ S}    : Maximum similarity with any already selected item

    λ (lambda) ∈ [0, 1]:
        Controls trade-off between relevance and diversity

        λ → 1 : Focus on relevance (less diversity)
        λ → 0 : Focus on diversity (less relevance)
        λ ≈ 0.7 : Good balance in practice

Intuition:
    - First item selected is the most relevant (since S is empty)
    - Subsequent items are chosen to be:
        * Similar to the query (high relevance)
        * Dissimilar to already selected items (low redundancy)

Why MMR:
    - Prevents recommending near-duplicate items
    - Improves diversity of recommendations
    - Produces more useful and realistic recommendation lists

In this project:
    - Sim(q, d) is computed using cosine similarity on TF-IDF vectors
    - MMR is applied on-demand (no full similarity matrix computed)
"""

"\nMaximal Marginal Relevance (MMR)\n--------------------------------\n\nMMR is a ranking technique used to select items that are both:\n1. Relevant to a query\n2. Diverse from each other\n\nIt is commonly used in recommendation systems, search engines, and information retrieval\nto avoid returning near-duplicate results.\n\nFormula:\n    MMR(d) = λ * Sim(q, d) - (1 - λ) * max_{d' ∈ S} Sim(d, d')\n\nWhere:\n    q  : Query item (e.g., product user interacted with)\n    d  : Candidate item being evaluated\n    S  : Set of already selected/recommended items\n\n    Sim(q, d)       : Similarity between query and candidate (relevance)\n    Sim(d, d')      : Similarity between candidate and selected items (redundancy)\n    max_{d' ∈ S}    : Maximum similarity with any already selected item\n\n    λ (lambda) ∈ [0, 1]:\n        Controls trade-off between relevance and diversity\n\n        λ → 1 : Focus on relevance (less diversity)\n        λ → 0 : Focus on diversity (less relevance)\n        λ

In [65]:
# Fast Lookup
product_to_idx = pd.Series(df.index, index=df["Product_Name"]).to_dict()

In [68]:
def MMR_recommend(product_name, top_n = 5, lambda_param = 0.7):

    idx = product_to_idx.get(product_name)
    if idx is None:
        print("Product not found!!!")
        return None
    
    sim_to_query = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()

    selected = []
    candidates = list(range(len(df)))
    candidates.remove(idx)

    top_candidates = np.argpartition(sim_to_query, -200)[-200:]
    candidates = [c for c in top_candidates if c != idx]

    for _ in range(top_n):
        
        mmr_scores = []

        for c in candidates:
            
            relevance = sim_to_query[c]

            if relevance > 0.95:
                continue

            if len(selected) == 0:
                redundancy = 0

            else:
                sim_to_selected = cosine_similarity(tfidf_matrix[c], tfidf_matrix[selected]).flatten()
                redundancy = sim_to_selected.max()

            score = lambda_param * relevance - (1 - lambda_param) * redundancy

            mmr_scores.append((c, score))
        
        if not mmr_scores:
            break
        
        best_idx = sorted(mmr_scores, key = lambda x: x[1], reverse=True)[0][0]

        selected.append(best_idx)
        candidates.remove(best_idx)

    results = df.iloc[selected][["Product_Name", "Category"]].copy()
    results["similarity_score"] = [sim_to_query[i] for i in selected]

    return results

In [70]:
product = "ROVSUN Ice Maker Machine Countertop, Make 44lbs Ice in 24 Hours, Compact & Portable Ice Maker with Ice Basket for Home, Office, Kitchen, Bar (Silver)"

result = MMR_recommend(product, top_n=5, lambda_param=0.5)
print(result)

                                            Product_Name  \
25597  ROVSUN Ice Maker Machine Countertop, Make 44lb...   
66708  Rainbow Tree Automatic Electric Ice Maker 26 P...   
3382   ZAFRO Ice Maker Countertop, Portable Ice Maker...   
26468  ROVSUN Ice Maker Machine Countertop, Make 26lb...   
73662  Ice Maker Machine Countertop, 3 in 1 Portable ...   

                      Category  similarity_score  
25597  Industrial & Scientific          0.916382  
66708               Appliances          0.486147  
3382                Appliances          0.468028  
26468  Industrial & Scientific          0.865291  
73662  Industrial & Scientific          0.489222  
